In [1]:
!pip install -q -U transformers accelerate bitsandbytes peft trl qwen-vl-utils pillow==10.4.0 "gym<=0.25.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.5/734.5 kB 14.4 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 68.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 20.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━

In [2]:
"""
datasets.py: Module xử lý dữ liệu, tokenize và tiền xử lý cho mô hình MSIT
(Multimodal Self-Correction Instruction Tuning for Open-World Product Attribute Mining - ACL 2025).

Bao gồm:
1. AGTD_INSTRUCTIONS & CTTD_INSTRUCTIONS.
2. Raw Dataset Loaders (WOAM, OAMine).
3. MSITDataPreparator: Bộ tiền xử lý (Lọc nhiễu, Cân bằng danh mục, Chia tập, Mẫu đối lập).
4. PyTorch Dataset Classes (AGTD, CTTD, Joint).
5. Multimodal Data Collator.
"""

import os
import sys
import json
import random
from typing import List, Dict, Any, Optional, Union, Tuple
from dataclasses import dataclass

from qwen_vl_utils import process_vision_info

if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

try:
    from PIL import Image
except ImportError:
    Image = None

try:
    import torch
    from torch.utils.data import Dataset
except ImportError:
    class Dataset:
        pass


# ==============================================================================
# 1. DANH SÁCH PROMPTS / INSTRUCTIONS CHUẨN TỪ BÀI BÁO (TABLE 4 & TABLE 5)
# ==============================================================================

AGTD_INSTRUCTIONS: List[str] = [
    "Extract the information from the title, bullet points, and product picture into JSON format.",
    "Convert the attribute values of the product from the provided information into JSON format.",
    "Generate the product attribute values in JSON format based on the provided title, bullet points, and picture.",
    "Compile the product's characteristic attributes into JSON format according to the provided information.",
    "Extract the product attribute information into JSON format from the provided title, bullet points, and picture.",
    "Parse the product's features and attributes into JSON format from the given information.",
    "Extract the product's characteristics into JSON format using the provided title, bullet points, and picture.",
    "Retrieve and organize the product's attribute values into JSON format from the provided information.",
    "Compile the product's attribute information into JSON format based on the title, bullet points, and picture content.",
    "Generate JSON-formatted product attribute data based on the provided information."
]

CTTD_INSTRUCTIONS: List[str] = [
    "I provide you with text and image information of a product along with one attribute of this product. Please determine if this product possesses this attribute through the text and image. Let's think step by step, and please provide your reasoning process.",
    "Here's textual and visual data about a product, along with a specific attribute. Your job is to discern if this attribute applies to the product, using both the text and the visuals. Let's methodically analyze the information, detailing your reasoning process step by step.",
    "I present you with textual and visual data about a product, along with a single attribute associated with it. Your task is to determine whether this product exhibits this attribute, utilizing both the text and the image. Let's think step by step, and please provide your reasoning process.",
    "I provide you with the text and picture information of a product and an attribute of this product, please help me judge whether this product has this attribute through the text and picture, we will think step by step, please give your reasoning process.",
    "I will give you the text and image information of a product, as well as one of its attributes. Please use the text and image to help me determine whether the product has this attribute. Let's think step by step, and please explain your reasoning process."
]

COT_5STEP_GUIDE = (
    "Five-step Chain of thought:\n"
    "Step 1: Determine the type of product based on pictures and text information.\n"
    "Step 2: Analyze the meaning of the attribute and judge whether the attribute matches the product type based on common sense.\n"
    "Step 3: Check if the product exhibits this attribute from the image.\n"
    "Step 4: Check if the product exhibits this attribute from the text description and verify exact value if present.\n"
    "Step 5: Summarize the conclusions from the above steps and draw the final decision (yes or no).\n"
)


# ==============================================================================
# 2. BỘ ĐỌC DỮ LIỆU GỐC (RAW DATA READERS)
# ==============================================================================

class WOAMRawReader:
    def __init__(self, file_path: str):
        self.file_path = file_path
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Không tìm thấy file WOAM: {file_path}")

    def load_records(self, category: Optional[str] = None, split: Optional[str] = None, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        records = []
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                item = json.loads(line)

                if category and item.get("pt", "").upper() != category.upper():
                    continue

                is_dev, is_test = item.get("is_dev", False), item.get("is_test", False)
                if split == "dev" and not is_dev: continue
                elif split == "test" and not is_test: continue
                elif split == "train" and (is_dev or is_test): continue

                title = item.get("title", "").strip()
                bullet_points = item.get("bullet_point", [])
                bullet_text = " ".join([str(bp).strip() for bp in bullet_points if bp]) if isinstance(bullet_points, list) else str(bullet_points).strip()

                records.append({
                    "asin": item.get("asin", ""),
                    "category": item.get("pt", ""),
                    "title": title,
                    "bullet_point": bullet_text,
                    "raw": item
                })
                if limit and len(records) >= limit:
                    break
        return records

class OAMineRawReader:
    def __init__(self, base_dir: str):
        self.base_dir = base_dir
        self.annotations_dir = os.path.join(base_dir, "annotations")
        self.raw_dir = os.path.join(base_dir, "raw")

    def list_categories(self, use_annotations: bool = True) -> List[str]:
        target_dir = self.annotations_dir if use_annotations else self.raw_dir
        if not os.path.exists(target_dir): return []
        return [f.replace(".jsonl", "") for f in os.listdir(target_dir) if f.endswith(".jsonl")]

    def load_annotated_records(self, category: Optional[str] = None, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        records = []
        files = [f"{category}.jsonl"] if category else os.listdir(self.annotations_dir)

        for filename in files:
            if not filename.endswith(".jsonl"): continue
            cat_name = filename.replace(".jsonl", "")
            filepath = os.path.join(self.annotations_dir, filename)
            if not os.path.exists(filepath): continue

            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    item = json.loads(line)
                    records.append({
                        "asin": item.get("asin", ""),
                        "category": cat_name,
                        "title": item.get("title", "").strip(),
                        "bullet_point": item.get("bullet_points", item.get("description", "")),
                        "raw": item
                    })
                    if limit and len(records) >= limit:
                        return records
        return records


# ==============================================================================
# 3. BỘ TIỀN XỬ LÝ VÀ CHIA TẬP (DATA PREPARATOR)
# ==============================================================================

class MSITDataPreparator:
    """
    Tiền xử lý và chia tập chuẩn xác theo paper MSIT:
    1. Lọc Modality Noise (Kiểm tra ảnh tồn tại + Độ dài văn bản >= 10 từ).
    2. Cân bằng danh mục (Rút đều số lượng từ các danh mục).
    3. Ngăn chặn Data Leakage (Tách biệt hoàn toàn Train/Test).
    4. Contrastive Sampling cho CTTD (50% Positive, 50% Negative).
    """
    def __init__(self, records: List[Dict], image_dir: str):
        self.records = records
        self.image_dir = image_dir

    def _filter_noise(self) -> List[Dict]:
        valid_records = []
        for r in self.records:
            # Nguyên tắc 4: Lọc Modality Noise
            img_path = os.path.join(self.image_dir, f"{r['asin']}.jpg")
            if not os.path.exists(img_path):
                continue
            
            text_length = len(r['title'].split()) + len(r['bullet_point'].split())
            if text_length < 10:
                continue
                
            r['image_path'] = img_path
            valid_records.append(r)
        return valid_records

    def prepare_splits(self) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        cleaned_records = self._filter_noise()
        
        # Nhóm theo category để cân bằng
        cat_dict = {}
        for r in cleaned_records:
            cat = r['category']
            if cat not in cat_dict:
                cat_dict[cat] = []
            cat_dict[cat].append(r)

        for cat in cat_dict:
            random.shuffle(cat_dict[cat])

        # Rút đều (Round-robin) để đảm bảo Category Balancing
        balanced_pool = []
        while cat_dict:
            for cat in list(cat_dict.keys()):
                if cat_dict[cat]:
                    balanced_pool.append(cat_dict[cat].pop(0))
                else:
                    del cat_dict[cat]

        # Kiểm tra đủ số lượng 2300 mẫu (1000 Test + 1000 AGTD + 300 CTTD)
        if len(balanced_pool) < 2300:
            print(f"[Cảnh báo] Chỉ có {len(balanced_pool)} mẫu hợp lệ, ít hơn yêu cầu 2300 của paper.")
            
        test_pool = balanced_pool[:1000]
        agtd_pool = balanced_pool[1000:2000]
        cttd_pool = balanced_pool[2000:2300]

        # Nguyên tắc 3: Lấy mẫu đối lập (Contrastive Sampling) cho CTTD
        # 150 Positive, 150 Negative
        all_categories = list(set(r['category'] for r in cleaned_records))
        for i, item in enumerate(cttd_pool):
            if i < len(cttd_pool) // 2:
                item['cttd_label'] = "yes"
                item['attribute_to_judge'] = "[PLACEHOLDER_POSITIVE: Điền thuộc tính có thật từ raw data]"
            else:
                item['cttd_label'] = "no"
                other_cats = [c for c in all_categories if c != item['category']]
                random_fake_cat = random.choice(other_cats) if other_cats else "Khác"
                item['attribute_to_judge'] = f"[PLACEHOLDER_NEGATIVE: Điền thuộc tính giả mạo lấy từ danh mục {random_fake_cat}]"

        return test_pool, agtd_pool, cttd_pool


# ==============================================================================
# 4. CÁC LỚP PYTORCH DATASET
# ==============================================================================

def load_image_safely(image_source: Optional[Union[str, Any]], image_dir: Optional[str] = None) -> Optional[Any]:
    if Image is None: return None
    img_path = None
    if isinstance(image_source, str) and image_source:
        if os.path.exists(image_source): img_path = image_source
        elif image_dir:
            candidate = os.path.join(image_dir, f"{image_source}.jpg")
            if os.path.exists(candidate): img_path = candidate
            
    if img_path and os.path.exists(img_path):
        try: return Image.open(img_path).convert("RGB")
        except Exception: pass
    return Image.new("RGB", (224, 224), color=(240, 240, 240))

class MSITAGTDDataset(Dataset):
    def __init__(self, jsonl_path: str, image_dir: Optional[str] = None, use_random_instruction: bool = True):
        self.jsonl_path = jsonl_path
        self.image_dir = image_dir
        self.instructions = AGTD_INSTRUCTIONS
        self.use_random_instruction = use_random_instruction
        self.samples = self._load_data()

    def _load_data(self) -> List[Dict[str, Any]]:
        samples = []
        if not os.path.exists(self.jsonl_path): return samples
        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip(): samples.append(json.loads(line))
        return samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.samples[idx]
        instruction = random.choice(self.instructions) if self.use_random_instruction else self.instructions[0]

        title = item.get("title", "").strip()
        bullet_points = item.get("bullet_point", "")
        if isinstance(bullet_points, list):
            bullet_points = "\n".join([f"- {bp}" for bp in bullet_points if bp])

        user_content = f"<image>\n{instruction}\n\nTitle: {title}\nBullet points: {bullet_points}"
        
        attributes = item.get("attributes", item.get("merged_attributes", {}))
        target_text = json.dumps(attributes, ensure_ascii=False, indent=2) if isinstance(attributes, dict) else str(attributes).strip()
        
        img_id = item.get("image_path") or item.get("asin")
        return {
            "type": "agtd",
            "asin": item.get("asin", ""),
            "image": load_image_safely(img_id, self.image_dir),
            "prompt": user_content,
            "target": target_text,
            "category": item.get("category", "")
        }

class MSITCTTDDataset(Dataset):
    def __init__(self, jsonl_path: str, image_dir: Optional[str] = None, use_random_instruction: bool = True):
        self.jsonl_path = jsonl_path
        self.image_dir = image_dir
        self.instructions = CTTD_INSTRUCTIONS
        self.use_random_instruction = use_random_instruction
        self.samples = self._load_data()

    def _load_data(self) -> List[Dict[str, Any]]:
        samples = []
        if not os.path.exists(self.jsonl_path): return samples
        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip(): samples.append(json.loads(line))
        return samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.samples[idx]
        instruction = random.choice(self.instructions) if self.use_random_instruction else self.instructions[0]

        attribute_to_judge = item.get("attribute_to_judge", "Unknown")
        title = item.get("title", "").strip()
        bullet_points = item.get("bullet_point", "")
        if isinstance(bullet_points, list):
            bullet_points = "\n".join([f"- {bp}" for bp in bullet_points if bp])

        user_content = f"<image>\n{instruction}\n\nThe attribute I want to judge is: {attribute_to_judge}\nTitle: {title}\nBullet Point: {bullet_points}"

        if "cot_response" in item:
            target_text = item["cot_response"]
        else:
            decision = item.get("cttd_label", item.get("decision", "yes")).strip().capitalize()
            target_text = (
                "Step 1: Product type analyzed.\n"
                "Step 2: Common sense reasoning performed.\n"
                "Step 3: Image validation completed.\n"
                "Step 4: Text verification completed.\n"
                f"Step 5: Based on the above steps, the attribute is valid.\n\n{decision}!"
            )

        img_id = item.get("image_path") or item.get("asin")
        return {
            "type": "cttd",
            "asin": item.get("asin", ""),
            "image": load_image_safely(img_id, self.image_dir),
            "prompt": user_content,
            "target": target_text,
            "attribute": attribute_to_judge,
            "decision": item.get("cttd_label", "yes")
        }

class MSITJointDataset(Dataset):
    def __init__(self, agtd_dataset: MSITAGTDDataset, cttd_dataset: MSITCTTDDataset, shuffle: bool = True):
        self.agtd_dataset = agtd_dataset
        self.cttd_dataset = cttd_dataset
        self.indices = [("agtd", i) for i in range(len(agtd_dataset))] + [("cttd", j) for j in range(len(cttd_dataset))]
        if shuffle: random.shuffle(self.indices)

    def __len__(self) -> int: return len(self.indices)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        dtype, sub_idx = self.indices[idx]
        return self.agtd_dataset[sub_idx] if dtype == "agtd" else self.cttd_dataset[sub_idx]


# ==============================================================================
# 5. MULTIMODAL DATA COLLATOR (TỐI ƯU CHO QWEN2-VL)
# ==============================================================================

@dataclass
class MSITDataCollator:
    processor: Any
    max_length: int = 2048
    ignore_index: int = -100

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        texts = []
        all_image_inputs = []
        prompt_lengths = []

        for item in batch:
            image = item["image"]
            p = item["prompt"]
            t = item["target"]
            
            # Loại bỏ chuỗi <image> giả bị hardcode
            p_clean = p.replace("<image>\n", "").replace("<image>", "").strip()

            # 1. Cấu trúc messages chuẩn của Qwen2-VL
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": p_clean}
                    ]
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": t}
                    ]
                }
            ]

            # 2. Xử lý Template (Tự động sinh ra đủ số lượng <|image_pad|> khớp với ảnh)
            prompt_messages = [messages[0]]
            full_text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            prompt_only = self.processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
            texts.append(full_text)
            
            # 3. Trích xuất features từ ảnh
            image_inputs, _ = process_vision_info(messages)
            if image_inputs:
                all_image_inputs.extend(image_inputs)
            
            # 4. Tính toán độ dài prompt (để mask -100)
            # Tokenize prompt riêng biệt để đo chiều dài chính xác
            prompt_tokens = self.processor.tokenizer(prompt_only, add_special_tokens=False)["input_ids"]
            prompt_lengths.append(len(prompt_tokens))

        # Đóng gói qua Processor của Qwen
        batch_encoding = self.processor(
            text=texts,
            images=all_image_inputs if all_image_inputs else None,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = batch_encoding["input_ids"]
        labels = input_ids.clone()
        pad_token_id = self.processor.tokenizer.pad_token_id

        # Mask nhãn -100 cho toàn bộ Prompt và Padding (Mô hình chỉ học phần câu trả lời của Assistant)
        for i, p_len in enumerate(prompt_lengths):
            labels[i, :min(p_len, labels.shape[1])] = self.ignore_index
            
            if pad_token_id is not None:
                labels[i][input_ids[i] == pad_token_id] = self.ignore_index

        batch_encoding["labels"] = labels
        return batch_encoding

# ==============================================================================
# 6. HÀM TIỆN ÍCH KHỞI TẠO TẬP HUẤN LUYỆN
# ==============================================================================

def build_msit_datasets(
    agtd_file: str,
    cttd_file: str,
    image_dir: Optional[str] = None
) -> Tuple[MSITAGTDDataset, MSITCTTDDataset, MSITJointDataset]:
    """
    Hàm khởi tạo nhanh 3 bộ Dataset: AGTD, CTTD và Joint Dataset theo chuẩn MSIT.
    """
    agtd_ds = MSITAGTDDataset(jsonl_path=agtd_file, image_dir=image_dir)
    cttd_ds = MSITCTTDDataset(jsonl_path=cttd_file, image_dir=image_dir)
    
    # Kết hợp 2 tập dữ liệu và xáo trộn (shuffle) cho quá trình Joint Instruction Tuning
    joint_ds = MSITJointDataset(agtd_dataset=agtd_ds, cttd_dataset=cttd_ds, shuffle=True)
    
    return agtd_ds, cttd_ds, joint_ds

In [3]:
import os
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Import từ module datasets.py của bạn (Đảm bảo file datasets.py nằm trong thư mục làm việc)
# Nếu file nằm trong thư mục input của Kaggle, bạn cần thêm đường dẫn vào sys.path
import sys
sys.path.append("/kaggle/input/datasets/kunzero106/msit-checkpoint-files/results/results") # Sửa lại tên thư mục chứa file datasets.py

# Cấu hình đường dẫn
BASE_DIR = "/kaggle/input/datasets/kunzero106/msit-checkpoint-files/results/results"
AGTD_FILE = os.path.join(BASE_DIR, "agtd_train.jsonl")
CTTD_FILE = os.path.join(BASE_DIR, "cttd_train.jsonl")
IMAGE_DIR = os.path.join(BASE_DIR, "dataset_images")

# Khởi tạo 3 tập dữ liệu
agtd_ds, cttd_ds, joint_ds = build_msit_datasets(
    agtd_file=AGTD_FILE, 
    cttd_file=CTTD_FILE, 
    image_dir=IMAGE_DIR
)

print(f"Tổng số mẫu huấn luyện Joint Dataset: {len(joint_ds)}")

Tổng số mẫu huấn luyện Joint Dataset: 1300


In [4]:
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

# 1. Cấu hình lượng tử hóa 4-bit (Tiết kiệm VRAM, nén mô hình 7B xuống ~5GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Đang tải Processor và Base Model...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 2. Chuẩn bị mô hình cho k-bit training (Gradient Checkpointing)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False # Bắt buộc tắt khi dùng gradient checkpointing

# 3. Cấu hình LoRA
# Nhắm mục tiêu vào các lớp Attention để mô hình học cách liên kết đặc trưng đa phương thức
lora_config = LoraConfig(
    r=16, # Rank (Độ phân giải của ma trận cập nhật, 16 là mức tiêu chuẩn tốt cho SFT)
    lora_alpha=32, # Hệ số tỷ lệ
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Gắn LoRA Adapter vào mô hình gốc
peft_model = get_peft_model(model, lora_config)

# In báo cáo số lượng tham số huấn luyện
peft_model.print_trainable_parameters()

Đang tải Processor và Base Model...


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

trainable params: 5,046,272 || all params: 8,296,421,888 || trainable%: 0.0608


In [ ]:
# Khởi tạo Collator
data_collator = MSITDataCollator(processor=processor, max_length=1024)

# Cấu hình siêu tham số huấn luyện tương đồng với các bài báo SFT đa phương thức
training_args = TrainingArguments(
    output_dir="/kaggle/working/msit_qwen_lora",
    num_train_epochs=10, 
    per_device_train_batch_size=2, 
    gradient_accumulation_steps=8, 
    learning_rate=3e-4, 
    warmup_steps=20,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit", 
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    fp16=True,
    report_to="none", 
    remove_unused_columns=False 
)

# Khởi tạo Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=joint_ds,
    data_collator=data_collator,
)

print("Bắt đầu tiến trình Instruction Tuning (SFT)...")
trainer.train()

# Lưu trọng số LoRA Adapter sau khi hoàn tất
trainer.save_model("/kaggle/working/msit_qwen_lora_final")
processor.save_pretrained("/kaggle/working/msit_qwen_lora_final")
print("Đã lưu mô hình tinh chỉnh thành công!")

In [5]:
import torch
import os
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
LORA_CHECKPOINT = "/kaggle/input/datasets/kunzero106/msit-checkpoint-files/checkpoint/msit_qwen_lora_final"

print("Đang nạp lại Base Model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Đang gắn trọng số LoRA (Checkpoint) vào mô hình...")
# Lệnh này sẽ ghi đè các ma trận q_proj, v_proj đã học được vào mô hình gốc
peft_model = PeftModel.from_pretrained(base_model, LORA_CHECKPOINT)

print("Đang nạp Processor...")
processor = AutoProcessor.from_pretrained(LORA_CHECKPOINT)

print("Phục hồi thành công! Sẵn sàng chạy Inference.")

Đang nạp lại Base Model (4-bit)...


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Đang gắn trọng số LoRA (Checkpoint) vào mô hình...
Đang nạp Processor...
Phục hồi thành công! Sẵn sàng chạy Inference.


In [7]:
import json
import re
from qwen_vl_utils import process_vision_info
from tqdm import tqdm

# Đảm bảo mô hình ở chế độ suy luận (tắt dropout, tiết kiệm tính toán)
peft_model.eval()

def qwen_infer(prompt: str, image_path: str = None) -> str:
    """Hàm ném dữ liệu qua mô hình đã fine-tune (đang nằm trong RAM)."""
    content = []
    if image_path and os.path.exists(image_path):
        content.append({"type": "image", "image": image_path})
    content.append({"type": "text", "text": prompt})
    
    messages = [{"role": "user", "content": content}]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        generated_ids = peft_model.generate(**inputs, max_new_tokens=512, temperature=0.1)
        
    trimmed_ids = generated_ids[0][len(inputs.input_ids[0]):]
    return processor.decode(trimmed_ids, skip_special_tokens=True).strip()

def run_self_correction_pipeline(test_file: str, image_dir: str, limit: int = 50):
    """Chạy quy trình MSIT Self-Correction: AGTD -> CTTD -> Lọc Yes/No"""
    print(f"Đang tải {limit} mẫu từ Test Set để chạy Pipeline...")
    
    with open(test_file, "r", encoding="utf-8") as f:
        test_samples = [json.loads(line) for line in f][:limit]

    final_schema = [] # Lưu trữ các thuộc tính hợp lệ (Pass CTTD)
    
    for idx, item in enumerate(tqdm(test_samples, desc="Xử lý Self-Correction")):
        title = item.get("title", "")
        bp = item.get("bullet_point", "")
        img_path = os.path.join(image_dir, f"{item['asin']}.jpg")
        
        # 1. GIAI ĐOẠN AGTD: Khai thác thuộc tính thô
        agtd_prompt = (
            "Extract product attributes and their values from the text and image.\n"
            f"Title: {title}\nBullet points: {bp}\n\n"
            "Output strictly in JSON format. Ensure all JSON keys are in lowercase snake_case format."
        )
        agtd_response = qwen_infer(agtd_prompt, img_path)
        
        # Tiền xử lý JSON
        clean_json = agtd_response.replace("```json", "").replace("```", "").strip()
        try:
            extracted_attrs = json.loads(clean_json)
        except:
            continue # Bỏ qua nếu mô hình xuất sai định dạng JSON
            
        # 2. GIAI ĐOẠN CTTD: Tự kiểm chứng từng thuộc tính
        validated_attrs = {}
        for attr_key, attr_val in extracted_attrs.items():
            cttd_prompt = (
                f"Evaluate if the product has the attribute: '{attr_key}'.\n"
                f"Title: {title}\nBullet points: {bp}\n\n"
                "Provide your reasoning in 5 steps (Product type -> Common sense -> Image -> Text -> Final decision yes/no).\n"
            )
            cttd_response = qwen_infer(cttd_prompt, img_path)
            
            # Kiểm tra xem từ cuối cùng trong bước 5 là Yes hay No
            if "yes" in cttd_response.lower().split("\n")[-1]:
                validated_attrs[attr_key] = attr_val
                final_schema.append(attr_key) # Thu thập khóa thuộc tính để gom cụm
        
        # Ghi nhận kết quả sau lọc
        item["msit_attributes"] = validated_attrs

    # Lưu kết quả pipeline
    output_path = "/kaggle/working/msit_pipeline_results.jsonl"
    with open(output_path, "w", encoding="utf-8") as f:
        for item in test_samples:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
    print(f"\nĐã lưu kết quả suy luận vào {output_path}")
    return list(set(final_schema)) # Trả về danh sách các keys duy nhất

# Thực thi Pipeline (Chạy thử 50 mẫu để xem kết quả)
TEST_FILE = "/kaggle/input/datasets/kunzero106/msit-checkpoint-files/results/results/test_set.jsonl"
unique_valid_attributes = run_self_correction_pipeline(TEST_FILE, IMAGE_DIR, limit=50)
print(f"Số lượng Key thuộc tính thô thu được: {len(unique_valid_attributes)}")

Đang tải 50 mẫu từ Test Set để chạy Pipeline...


Xử lý Self-Correction:   0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2356: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Xử lý Self-Correction: 100%|██████████| 50/50 [1:32:51<00:00, 111.43s/it]


Đã lưu kết quả suy luận vào /kaggle/working/msit_pipeline_results.jsonl
Số lượng Key thuộc tính thô thu được: 156


In [8]:
!pip install -q sentence-transformers scikit-learn

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def refine_attribute_schema(attribute_list: list, similarity_threshold: float = 0.85):
    """Sử dụng Sentence Transformer để tìm và gộp các thuộc tính đồng nghĩa."""
    if not attribute_list:
        return {}

    print("Đang tải mô hình nhúng (Embedding Model) để tính Cosine Similarity...")
    # Sử dụng mô hình miniLM siêu nhẹ và cực nhanh của HuggingFace
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Chuẩn hóa text (bỏ dấu gạch dưới) để nhúng chính xác hơn
    clean_attrs = [attr.replace("_", " ") for attr in attribute_list]
    embeddings = embedder.encode(clean_attrs)
    
    # Tính ma trận Cosine Similarity
    sim_matrix = cosine_similarity(embeddings)
    
    # Thuật toán gom cụm (Clustering)
    clusters = {}
    visited = set()
    
    for i in range(len(attribute_list)):
        if i in visited:
            continue
            
        current_attr = attribute_list[i]
        clusters[current_attr] = [current_attr]
        visited.add(i)
        
        for j in range(i + 1, len(attribute_list)):
            if j not in visited and sim_matrix[i][j] >= similarity_threshold:
                clusters[current_attr].append(attribute_list[j])
                visited.add(j)
                
    return clusters

# Thực thi gom cụm với ngưỡng tương đồng 85%
refined_schema = refine_attribute_schema(unique_valid_attributes, similarity_threshold=0.85)

print("\n=== KẾT QUẢ GOM CỤM THUỘC TÍNH (MSIT SCHEMA REFINEMENT) ===")
for root_attr, synonyms in refined_schema.items():
    if len(synonyms) > 1:
        print(f"• [Khóa chính]: {root_attr} ==> Gom nhóm từ: {synonyms}")
    else:
        print(f"• {root_attr}")

Đang tải mô hình nhúng (Embedding Model) để tính Cosine Similarity...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== KẾT QUẢ GOM CỤM THUỘC TÍNH (MSIT SCHEMA REFINEMENT) ===
• support
• absorption_method
• case_protection
• case_color
• storage
• fair_trade
• screen_protector
• pack_size
• paleo_diet_friendly
• bullet_points
• formula_contains
• first_purveyor
• natural_and_organic_flavoring
• case_features
• screen_protector_details
• dairy_free
• texture
• accessibility
• gifting
• quantity
• design_make_phone_stand_out
• gluten_free
• vegan
• compatible_model
• [Khóa chính]: packaging_material ==> Gom nhóm từ: ['packaging_material', 'packaging', 'packaging_materials']
• [Khóa chính]: case_design_details ==> Gom nhóm từ: ['case_design_details', 'case_design']
• anti_scratching
• range
• folic_acid
• [Khóa chính]: compatible_phone_model ==> Gom nhóm từ: ['compatible_phone_model', 'compatible_phone_model_max']
• days_supply
• certified_usda_organic
• description
• quality_standards
• [Khóa chính]: benefits ==> Gom nhóm từ: ['benefits', 'benefit']
• no_artificial_preservatives
• count
• brand
• su

In [18]:
import json
import re

def extract_spans(text: str, spans: list) -> list:
    """Hàm cắt chuỗi dựa trên tọa độ [start, end]"""
    extracted = []
    if not text or not spans:
        return extracted
        
    for span in spans:
        if isinstance(span, list) and len(span) == 2:
            try:
                start, end = int(span[0]), int(span[1])
                val = text[start:end].strip().lower()
                if val:
                    extracted.append(val)
            except:
                pass
    return extracted

def get_ground_truth_values(raw_item: dict) -> set:
    """Thu thập các Giá trị chuẩn bằng cách cắt văn bản theo tọa độ"""
    gt_values = set()
    
    title = raw_item.get("title", "")
    title_spans = raw_item.get("title_attr", [])
    if isinstance(title_spans, list) and len(title_spans) > 0:
        if isinstance(title_spans[0], list): 
            gt_values.update(extract_spans(title, title_spans))

    bp_list = raw_item.get("bullet_point", [])
    if isinstance(bp_list, str):
        try: bp_list = json.loads(bp_list)
        except: bp_list = [bp_list]
            
    bp_spans = raw_item.get("bp_attr", [])
    if isinstance(bp_spans, list) and isinstance(bp_list, list):
        for idx, spans_for_bp in enumerate(bp_spans):
            if idx < len(bp_list) and isinstance(spans_for_bp, list):
                gt_values.update(extract_spans(str(bp_list[idx]), spans_for_bp))
                
    return gt_values

def tokenize_to_words(text_or_list) -> set:
    """Băm nhuyễn chuỗi/list thành các từ vựng đơn lẻ, phá vỡ định dạng snake_case."""
    if isinstance(text_or_list, list):
        text_or_list = " ".join([str(v) for v in text_or_list])
    
    # Ép kiểu string, chữ thường, thay thế dấu _, - thành khoảng trắng
    clean_text = str(text_or_list).lower().replace('_', ' ').replace('-', ' ')
    # Dùng RegEx để nhặt ra tất cả các từ (chỉ chứa chữ cái/số)
    words = set(re.findall(r'\b\w+\b', clean_text))
    
    # Loại bỏ các stop words cơ bản không mang giá trị thuộc tính
    stop_words = {'and', 'or', 'with', 'the', 'a', 'an', 'of', 'for', 'in', 'to'}
    return words - stop_words

def calculate_token_level_metrics(gt_values: set, predicted_dict: dict) -> tuple:
    """Tính toán độ khớp theo cấp độ Từ vựng (Token-level Match)"""
    # 1. Băm Ground Truth thành các tokens
    gt_tokens = set()
    for val in gt_values:
        gt_tokens.update(tokenize_to_words(val))
        
    # 2. Băm Predicted Values thành các tokens (Bỏ chặn ignore_keys để vét sạch dữ liệu)
    pred_tokens = set()
    for v in predicted_dict.values():
        pred_tokens.update(tokenize_to_words(v))
            
    # 3. Tính toán Giao tập hợp
    true_positives = len(gt_tokens.intersection(pred_tokens))
    return true_positives, len(pred_tokens), len(gt_tokens)

def evaluate_msit_pipeline(results_file: str):
    total_tp, total_pred, total_gt = 0, 0, 0
    debug_count = 0
    
    print("Đang chạy Đánh giá cấp độ Từ vựng (Token-level Matching)...")
    
    with open(results_file, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            
            raw_data = item.get("raw", {})
            gt_values = get_ground_truth_values(raw_data)
            pred_attributes = item.get("msit_attributes", {})
            
            # Tính toán Token-level metrics
            tp, pred, gt = calculate_token_level_metrics(gt_values, pred_attributes)
            
            if debug_count < 2 and gt > 0:
                print(f"\n[DEBUG - MẪU #{debug_count + 1}]")
                print(f"Tokens Thực tế (GT)   : {set(tokenize_to_words(list(gt_values)))}")
                print(f"Tokens Mô hình sinh ra: {set(tokenize_to_words(list(pred_attributes.values())))}")
                print(f"-> Khớp đúng (TP)     : {tp} tokens")
                debug_count += 1
                
            total_tp += tp
            total_pred += pred
            total_gt += gt
            
    precision = total_tp / total_pred if total_pred > 0 else 0
    recall = total_tp / total_gt if total_gt > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print("\n=== KẾT QUẢ ĐÁNH GIÁ TOKEN-LEVEL MATCHING ===")
    print(f"Tổng số Tokens Dự đoán (Predicted) : {total_pred}")
    print(f"Tổng số Tokens Thực tế (GT)        : {total_gt}")
    print(f"Tổng số Tokens Khớp đúng (TP)      : {total_tp}")
    print("-" * 46)
    print(f"Precision (Độ chuẩn xác) : {precision * 100:.2f}%")
    print(f"Recall    (Độ bao phủ)   : {recall * 100:.2f}%")
    print(f"F1-Score  (Độ cân bằng)  : {f1_score * 100:.2f}%")

# Đường dẫn file sinh ra từ Cell 5
RESULTS_FILE = "/kaggle/working/msit_pipeline_results.jsonl"
evaluate_msit_pipeline(RESULTS_FILE)

Đang chạy Đánh giá cấp độ Từ vựng (Token-level Matching)...

[DEBUG - MẪU #1]
Tokens Thực tế (GT)   : {'500mg', 'skin', 'acne', 'metabolism', 'energy'}
Tokens Mô hình sinh ra: {'support', 'skin', 'supplement', 'acid', 'vegan', 'fatigue', 'vitamin', 'b', 'boost', '240', 'strength', '5', 'production', 'b5', '500mg', 'healthy', 'party', '3rd', 'high', 'tested', 'tiredness', 'acne', 'energy', 'supports', 'origins', 'premium', 'metabolism', 'pantothenic', 'capsules', 'cellular'}
-> Khớp đúng (TP)     : 5 tokens

[DEBUG - MẪU #2]
Tokens Thực tế (GT)   : {'kickstand', 'wallet', 'soft', 'case', 'snap', 'magnetic', 'leather', 'mandala', 'flip', 'tpu', 'women'}
Tokens Mô hình sinh ra: set()
-> Khớp đúng (TP)     : 0 tokens

=== KẾT QUẢ ĐÁNH GIÁ TOKEN-LEVEL MATCHING ===
Tổng số Tokens Dự đoán (Predicted) : 1334
Tổng số Tokens Thực tế (GT)        : 329
Tổng số Tokens Khớp đúng (TP)      : 157
----------------------------------------------
Precision (Độ chuẩn xác) : 11.77%
Recall    (Độ bao phủ)   